# Underwater YOLO11 — ***CBAM*** Held-out Test Evaluation (Colab Template v4)

This notebook:

- Installs **Ultralytics** from a **specific GitHub branch** in the fork
- Extracts `UOD_Dataset.tar` from **Google Drive** to Colab’s local storage
- Saves runs + checkpoints to **Google Drive**, so you can **resume** after idle timeouts

**Workflow:** To change `MODEL_CONFIG_YAML`, `ABLATION_NAME`, and `BRANCH_NAME` in the configuration cell for each ablation.

## 0) Configuration

Central settings for experiment naming, storage paths, training parameters, augmentations, and required hardware. Review these values before running each ablation.

In [ ]:
from google.colab import runtime
import albumentations as A
import torch
from pathlib import Path

# The ablation experiment identifier
ABLATION_NAME = "CBAM-detect"

# Set the git branch name to correspond to the ablation name
BRANCH_NAME = "ablation/attention-cbam-detect-p3-p5"
BASE_DIR = Path("/content/drive/MyDrive/UnderWaterYOLO11/")

# Set ultralytics project name to correspond to the ablation name
PROJECT_NAME = BASE_DIR / ABLATION_NAME

# Set the scale to be used in this ablation
SCALE = 'm'

RUN_NAME = f'train-{SCALE}'
TEST_RUN_NAME = f'test-{SCALE}'

DATASET_ROOT = Path("/content/UOD_Dataset")
DATA_YAML = DATASET_ROOT / "data_split.yaml"

# COCO annotation file for faster-coco-eval
COCO_ANN_FILE = BASE_DIR /"instances_test2017.json"

# Training parameters
IMG_SIZE = 640
BATCH    = 64

REQUIRED_GPU = 'NVIDIA RTX PRO 6000 Blackwell Server Edition' # 8.71 units per hour
# REQUIRED_GPU = 'NVIDIA A100-SXM4-40GB' # 5.44 units per hour
# REQUIRED_GPU = 'NVIDIA A100-SXM4-80GB' # 7.52 units per hour
# REQUIRED_GPU = None
gpu_name = torch.cuda.get_device_name(0)
if REQUIRED_GPU and gpu_name != REQUIRED_GPU:
    print('❌ This notebook must execute on the google Colab with runtime type: ', REQUIRED_GPU)
    runtime.unassign()
else:
    print('✅ Running on', gpu_name)

✅ Running on NVIDIA RTX PRO 6000 Blackwell Server Edition


## 1) Mount Google Drive and set a Drive-backed experiment directory
All training artifacts (including `last.pt`) are written to Google Drive.


In [ ]:
# 📂 Mount Google Drive
from google.colab import drive

drive.mount('/content/drive')

# Create base directory if it does not exist
BASE_DIR.mkdir(parents=True, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2) Install and import Ultralytics from fork + branch

In [ ]:
%pip install "git+https://github.com/marcofarrugia/ultralytics.git@{BRANCH_NAME}"

import ultralytics
from ultralytics import YOLO
from ultralytics.utils import YAML

ultralytics.checks()

Ultralytics 8.3.244 [Underwater YOLO11 Ablations (attention-cbam-detect-p3-p5) by Marco Farrugia] 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
Setup complete ✅ (48 CPUs, 176.9 GB RAM, 47.9/235.7 GB disk)


## 3) Extract the dataset from Google Drive

The dataset is extracted from `UOD_Dataset.tar` inside `BASE_DIR`, to Colab's local `/content` storage for faster access during training. The dataset directory and its `data_split.yaml` configuration file are identified for use by the remaining cells.

In [ ]:
import tarfile
import tempfile
from tqdm.auto import tqdm

DATASET_ARCHIVE = BASE_DIR / "UOD_Dataset.tar"

if not DATASET_ARCHIVE.is_file():
    raise FileNotFoundError(
        f"Dataset archive not found:\n{DATASET_ARCHIVE}"
    )

# Extract only when the dataset is not already available.
if not DATA_YAML.is_file():
    with tempfile.TemporaryDirectory(prefix="uod_dataset_") as temp_dir:
        copied_archive = Path(temp_dir) / DATASET_ARCHIVE.name
        archive_size = DATASET_ARCHIVE.stat().st_size

        print(f"Transferring dataset from: {DATASET_ARCHIVE}")

        # Copy the archive from Google Drive to temporary local storage.
        with (
            DATASET_ARCHIVE.open("rb") as source,
            copied_archive.open("wb") as destination,
            tqdm(
                total=archive_size,
                desc="Copying dataset",
                unit="B",
                unit_scale=True,
                unit_divisor=1024,
                dynamic_ncols=True,
            ) as progress,
        ):
            while chunk := source.read(8 * 1024 * 1024):
                destination.write(chunk)
                progress.update(len(chunk))

        # Extract from the temporary local copy.
        with tarfile.open(copied_archive, "r:*") as archive:
            members = archive.getmembers()
            archive.extractall(
                path=DATASET_ROOT.parent,
                members=tqdm(
                    members,
                    total=len(members),
                    desc="Extracting dataset",
                    unit="file",
                    dynamic_ncols=True,
                ),
                filter="data",
            )

if not DATA_YAML.is_file():
    raise FileNotFoundError(
        f"Dataset configuration file not found after extraction: {DATA_YAML}"
    )

print("Dataset extracted to:", DATASET_ROOT)
print("Dataset configuration:", DATA_YAML)

Transferring dataset from: /content/drive/MyDrive/UnderWaterYOLO11/UOD_Dataset.tar


Copying dataset:   0%|          | 0.00/4.15G [00:00<?, ?B/s]

Extracting dataset:   0%|          | 0/33256 [00:00<?, ?file/s]

Dataset extracted to: /content/UOD_Dataset
Dataset configuration: /content/UOD_Dataset/data_split.yaml


## 4) Generate COCO ground-truth annotations from the test split YOLO labels

Only the labels referenced by the test manifest from `DATA_YAML` are supplied to PyLabel, while the original image and label filenames remain unchanged. PyLabel assigns the numeric image IDs used by the COCO ground-truth file. Prediction IDs are aligned with these IDs by filename and written to a separate `predictions_coco_ids.json` file.

In [ ]:
%pip install "pylabel==0.1.55" -q --progress-bar on

from shutil import copy2
from tempfile import TemporaryDirectory
from pylabel import importer


if COCO_ANN_FILE.exists():
    print(f"COCO annotations already exist → skipping export:\n{COCO_ANN_FILE}")
else:
    COCO_ANN_FILE.parent.mkdir(parents=True, exist_ok=True)
    data_config = YAML.load(DATA_YAML)
    test_manifest = DATASET_ROOT / data_config["test"]
    print(f"Generating COCO annotations from YOLO labels: {test_manifest}")

    dataset_dir = test_manifest.parent
    labels_dir = dataset_dir / "labels"

    # PyLabel receives only labels belonging to the tes manifest.
    with TemporaryDirectory(
        prefix="coco_test_labels_",
        dir=dataset_dir,
    ) as temp_dir:
        temp_labels_dir = Path(temp_dir)

        for image_ref in test_manifest.read_text(encoding="utf-8").splitlines():
            label_name = f"{Path(image_ref).stem}.txt"
            copy2(
                labels_dir / label_name,
                temp_labels_dir / label_name,
            )

        importer.ImportYoloV5(
            path=str(temp_labels_dir),
            path_to_images="../images/",
            cat_names=data_config["names"],
            img_ext="jpg,jpeg,png",
        ).export.ExportToCoco(
            cat_id_index=1,
            output_path=str(COCO_ANN_FILE),
        )

    print(f"\nCOCO annotations written to: {COCO_ANN_FILE}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 70.8 MB/s eta 0:00:00
COCO annotations already exist → skipping export:
/content/drive/MyDrive/UnderWaterYOLO11/instances_test2017.json


## 5) Evaluate best checkpoint

In [ ]:
# Load best checkpoint
best_ckpt = PROJECT_NAME / RUN_NAME / "weights" / "best.pt"
best_model = YOLO(best_ckpt)

# Run test
test_results = best_model.val(
    data=DATA_YAML,
    imgsz=IMG_SIZE,
    batch=BATCH * 2, # A larger batch size potentially improves GPU utilisation and reduces per-batch overhead
    split="test",
    project=PROJECT_NAME,
    name=TEST_RUN_NAME,
    save_json=True,
    exist_ok=True,  # Replace any previous test results
    conf=0.001,
    iou=0.7,
    max_det=300,
    half=False,
    augment=False,
    verbose=True
)

Ultralytics 8.3.244 [Underwater YOLO11 Ablations (attention-cbam-detect-p3-p5) by Marco Farrugia] 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
YOLO11m-cbam-detect-p3-p5 summary (fused): 143 layers, 20,116,211 parameters, 0 gradients, 67.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 8204.9±4718.3 MB/s, size: 1065.9 KB)
val: Scanning /content/UOD_Dataset/dataset/labels... 1662 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1662/1662 4.2Kit/s 0.4s
val: New cache created: /content/UOD_Dataset/dataset/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 2.2it/s 5.8s
                   all       1662       9860      0.587      0.429       0.47       0.33
               bivalve        210       1095        0.4      0.234      0.247      0.165
                bottle        173        619      0.476      0.194      0.242      0.154
            cephal

## 6) Evaluate Test Results

### 6.1 Align Prediction IDs with COCO Ground Truth

Reference: [Ultralytics issue #21557 — incorrect `image_id` conversion](https://github.com/ultralytics/ultralytics/issues/21557)

In [ ]:
import json

save_dir = Path(test_results.save_dir)
pred_json = save_dir / "predictions.json"
aligned_pred_json = save_dir / "predictions_coco_ids.json"

# Map filenames to PyLabel-generated COCO image IDs.
with open(COCO_ANN_FILE, "r", encoding="utf-8") as f:
    coco_annotations = json.load(f)

coco_id_by_filename = {
    Path(image["file_name"]).name: image["id"]
    for image in coco_annotations["images"]
}

# Align every Ultralytics prediction with the PyLabel ground-truth image ID.
with open(pred_json, "r", encoding="utf-8") as f:
    predictions = json.load(f)

for prediction in predictions:
    filename = Path(prediction["file_name"]).name
    prediction["image_id"] = coco_id_by_filename[filename]

with open(aligned_pred_json, "w", encoding="utf-8") as f:
    json.dump(predictions, f)

### 6.2 Compute Test Metrics

In [ ]:

%pip install "faster-coco-eval==1.7.2" -q --progress-bar on

from faster_coco_eval import COCO, COCOeval_faster
from ultralytics.utils.torch_utils import get_flops, get_num_params

# -------------------------------
# Ultralytics metrics
# -------------------------------

# Inference timing
latency_inference = test_results.speed["inference"]

# Total inference pipeline latency per image, excluding validation-only loss computation
latency_end_to_end = sum(
    test_results.speed[key]
    for key in ("preprocess", "inference", "postprocess")
)

# Raw model speed
fps_inference = 1000.0 / latency_inference

# Real application throughput (including data handling)
fps_end_to_end = 1000.0 / latency_end_to_end

num_params = get_num_params(best_model.model)
flops = get_flops(best_model.model, imgsz=IMG_SIZE)

# -------------------------------
# COCO metrics
# -------------------------------
coco_gt = COCO(COCO_ANN_FILE)
coco_dt = coco_gt.loadRes(aligned_pred_json)

coco_eval = COCOeval_faster(
    coco_gt,
    coco_dt,
    iouType="bbox",
    print_function=lambda *args, **kwargs: None,
)
coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()

stats = coco_eval.stats_as_dict

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.9/587.9 kB 12.3 MB/s eta 0:00:00


### 6.3 Display Test Metrics

In [ ]:
metrics_order = [
    ("AP_all", "mAP@0.50:0.95"),
    ("AP_50", "mAP@0.50"),
    ("AP_75", "mAP@0.75"),
    ("AP_small", "mAP (Small)"),
    ("AP_medium", "mAP (Medium)"),
    ("AP_large", "mAP (Large)"),
    ("AR_third", "AR@0.50:0.95 (maxDets=100)"),
    ("AR_small", "AR (Small)"),
    ("AR_medium", "AR (Medium)"),
    ("AR_large", "AR (Large)"),
    # ("AR_all", "AR@0.50:0.95 (maxDets=1)"),
    # ("AR_second", "AR@0.50:0.95 (maxDets=10)"),
    # ("AR_50", "AR@0.50"),
    # ("AR_75", "AR@0.75")
]

# Pretty print the metrics
print("\n=============== TEST ACCURACY RESULTS ===============")
for key, label in metrics_order:
    value = stats.get(key, float("nan"))
    print(f"{label:<26} | {value:.4f}")
print("\n=============== TEST SPEED RESULTS ===============")
print(f"{'Inference Latency (forward pass only)':<37} | {latency_inference:.2f} ms / image")  # forward pass only, no pre/postprocessing
print(f"{'Inference Latency (end-to-end)':<37} | {latency_end_to_end:.2f} ms / image")        # preprocessing + inference + postprocessing
print(f"{'Inference Speed (forward pass only)':<37} | {fps_inference:.2f} FPS")
print(f"{'Inference Speed (end-to-end)':<37} | {fps_end_to_end:.2f} FPS")
print(f"{'Parameters':<37} | {num_params:,}")
print(f"{'GFLOPs':<37} | {flops:.1f}")
print("==================================================\n")



=============== TEST ACCURACY RESULTS ===============
mAP@0.50:0.95              | 0.3042
mAP@0.50                   | 0.4427
mAP@0.75                   | 0.3193
mAP (Small)                | 0.1133
mAP (Medium)               | 0.2392
mAP (Large)                | 0.3376
AR@0.50:0.95 (maxDets=100) | 0.4119
AR (Small)                 | 0.1940
AR (Medium)                | 0.3204
AR (Large)                 | 0.4431

=============== TEST SPEED RESULTS ===============
Inference Latency (forward pass only) | 1.40 ms / image
Inference Latency (end-to-end)        | 1.90 ms / image
Inference Speed (forward pass only)   | 713.87 FPS
Inference Speed (end-to-end)          | 525.28 FPS
Parameters                            | 20,116,211
GFLOPs                                | 67.8



## 7) Terminate the current runtime to preserve GPU Time

In [ ]:
print('✅ Mission accomplished')
runtime.unassign()

✅ Mission accomplished
